In [62]:
import os
import numpy as np
import pandas as pd

import requests
import pandas as pd
import io

import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection

import astropy.units as u
from astropy.coordinates import SkyCoord
from astropy.coordinates import angular_separation

import rocks
rocks.set_log_level("error")
import ssptools
import fink_utils.sso.spins as finkus

In [63]:
import figure_setup as fs

In [64]:
target = "223"
# target = "6826"

# Get SSOFT

In [65]:
# Local Configuration
data_fink = "/data/fink/spins/"
bft_file = os.path.join(data_fink, "data", "ssoBFT-latest.parquet")

In [66]:
# Threshold for selection (of non-zero values)
thres = 1e-3

# Minimum phase angle to consider
min_phase = 3

In [67]:
# ZTF filters 1: g, 2: r
filters = {"1": "g", "2": "r"}

fink_colors = ["#15284F", "#F5622E"]

V_minus_g = 0
V_minus_r = 0

In [68]:
data = pd.read_parquet(os.path.join(data_fink, "data", "ztf", "sso_ZTF.parquet"))

## Get Observations

In [69]:
# Get data from FINK
r = requests.post(
    "https://fink-portal.org/api/v1/sso",
    json={"n_or_d": target, "withEphem": True, "output-format": "json"},
)

# Format output in a DataFrame
ztf = pd.read_json(io.BytesIO(r.content))

In [70]:
# Compute Ephemerides
jd_min = ztf["Date"].min()-50
jd_max = ztf["Date"].max()+50
step = 2
nbd = (jd_max - jd_min) / step

eph = ssptools.ephemcc(target, ep=jd_min, nbd=nbd, step=f"{step}d")

In [71]:
# From apparent magnitude to unit distance
def dist_reduction(d_obs, d_sun):
    return 5 * np.log10(d_obs * d_sun)

## Plot for the article

In [86]:
# Mag vs Time
fig, ax = plt.subplots(
    3,
    1,
    # figsize=(12, 12), 
    figsize=fs.figsize(1),
    sharex=True,
    gridspec_kw={
        "top": 0.995,
        "left": 0.075,
        "right": 0.995,
        "bottom": 0.085,
        "hspace": 0.02,
        "height_ratios": [2, 2, 1],
    },
)
# ZTF Colors and Colors
colors = ["#15284F", "#F5622E"]

show_HG = True
show_HG1G2 = True
show_sHG1G2 = True

res_HG = False
res_HG1G2 = True
res_sHG1G2 = True

show_HG1G2sp = False
res_HG1G2sp = False

marker_size = 10


for fi, filt in enumerate(["g", "r"]):
    # --------------------------------------------------------------------------------
    # Plot ZTF observations
    cond = ztf["i:fid"] == fi + 1
    if filt == "g":
        color_corr = V_minus_g
    else:
        color_corr = V_minus_r

    # ax[fi].errorbar(
    #     ztf.loc[cond, "Date"],
    #     ztf.loc[cond, "i:magpsf"],
    #     yerr=ztf.loc[cond, "i:sigmapsf"],
    #     # s=10,
    #     marker=".",
    #     linestyle='None', 
    #     color=colors[fi],
    #     label=f"ZTF {filt}",
    # )
    ax[fi].scatter(
        ztf.loc[cond, "Date"],
        ztf.loc[cond, "i:magpsf"],
        s=marker_size,
        marker="o",
        color=colors[fi],
        label=f"ZTF {filt}",
    )

    # --------------------------------------------------------------------------------
    # HG model
    # Get Parameters
    H = data.loc[data.ssnamenr == target, "HG_H_{}".format(filt)].values[0]
    G = data.loc[data.ssnamenr == target, "HG_G_{}".format(filt)].values[0]
    if fi == 0:
        print(f"HG       : {H:5.2f}  {G:4.2f}")

    # Linear Plot
    if show_HG:
        pred_mag = finkus.func_hg(np.radians(eph["Phase"]), H, G) + dist_reduction(
            eph["Dobs"], eph["Dhelio"]
        )
        ax[fi].plot(
            eph["Date"], pred_mag, color=colors[fi], linestyle="dotted",
            label=r"$\texttt{HG}$",
        )

    # Plot residuals
    if res_HG:
        phase = ztf["Phase"]
        pred_mag = finkus.func_hg(np.radians(phase), H, G)
        ax[2].scatter(
            ztf.loc[:, "Date"],
            ztf.loc[:, "i:magpsf"]
            - dist_reduction(ztf.loc[:, "Dobs"], ztf.loc[:, "Dhelio"])
            - pred_mag,
            color=colors[fi],
            alpha=0.1,
            marker="s",
            s=marker_size,
            label=f"HG",
            rasterized=True
        )

    # --------------------------------------------------------------------------------
    # HG1G2 model
    # Get Parameters
    H = data.loc[data.ssnamenr == target, "HG1G2_H_{}".format(filt)].values[0]
    G1 = data.loc[data.ssnamenr == target, "HG1G2_G1_{}".format(filt)].values[0]
    G2 = data.loc[data.ssnamenr == target, "HG1G2_G2_{}".format(filt)].values[0]

    if fi == 0:
        print(f"HG1G2    : {H:5.2f}  {G1:4.2f}  {G2:4.2f}")

    # Linear Plot
    if show_HG1G2:
        pred_mag = finkus.func_hg1g2(
            np.radians(eph["Phase"]), H, G1, G2
        ) + dist_reduction(eph["Dobs"], eph["Dhelio"])
        ax[fi].plot(
            eph["Date"], pred_mag, color=colors[fi], linestyle="dashed",
            label=r"$\texttt{HG}_{\texttt{1}}\texttt{G}_{\texttt{2}}$"
            )

    # Plot Residuals
    if res_HG1G2:
        pred_mag = finkus.func_hg1g2(np.radians(ztf.loc[cond, "Phase"]), H, G1, G2)
        ax[2].scatter(
            ztf.loc[cond, "Date"],
            ztf.loc[cond, "i:magpsf"]
            - dist_reduction(ztf.loc[cond, "Dobs"], ztf.loc[cond, "Dhelio"])
            - pred_mag,
            color=colors[fi],
            alpha=0.1,
            marker="s",
            s=marker_size, 
            label=r"$\texttt{HG}_{\texttt{1}}\texttt{G}_{\texttt{2}}$"+ f" {filt}",
            rasterized=True
        )


    # --------------------------------------------------------------------------------
    # TEST: HG1G2 model with shg1g2 parameters
    # Get Parameters
    # H = data.loc[data.ssnamenr == target, "SHG1G2_H_{}".format(filt)].values[0]
    # G1 = data.loc[data.ssnamenr == target, "SHG1G2_G1_{}".format(filt)].values[0]
    # G2 = data.loc[data.ssnamenr == target, "SHG1G2_G2_{}".format(filt)].values[0]

    # # Linear Plot
    # if show_HG1G2:
    #     pred_mag = finkus.func_hg1g2(
    #         np.radians(eph["Phase"]), H, G1, G2
    #     ) + dist_reduction(eph["Dobs"], eph["Dhelio"])
    #     ax[fi].plot(
    #         eph["Date"], pred_mag, color='grey', linestyle="dashed",
    #         label=r"$\texttt{HG}_{\texttt{1}}\texttt{G}_{\texttt{2}}$"
    #         )






    # --------------------------------------------------------------------------------
    # SHG1G2 model
    # Get Parameters
    H = data.loc[data.ssnamenr == target, f"SHG1G2_H_{filt}"].values[0]
    G1 = data.loc[data.ssnamenr == target, f"SHG1G2_G1_{filt}"].values[0]
    G2 = data.loc[data.ssnamenr == target, f"SHG1G2_G2_{filt}"].values[0]
    ra0 = data.loc[data.ssnamenr == target, "SHG1G2_alpha0"].values[0]
    dec0 = data.loc[data.ssnamenr == target, "SHG1G2_delta0"].values[0]
    R = data.loc[data.ssnamenr == target, "SHG1G2_R"].values[0]
    rms = data.loc[data.ssnamenr == target, "SHG1G2_rms"].values[0]
    print(
        f"HG21G1  {filt}: {H:5.2f}  {G1:4.2f}  {G2:4.2f}   {ra0:4.1f}  {dec0:4.1f}   {R:4.2f}    {rms:.3f}"
    )

    # Plot
    if show_sHG1G2:
        radec = SkyCoord(eph.RA, eph.DEC, unit=(u.hourangle, u.deg))
        pha = np.transpose(
            [
                [i, j, k]
                for i, j, k in zip(
                    np.radians(eph.Phase),
                    np.radians(radec.ra.deg),
                    np.radians(radec.dec.deg),
                )
            ]
        )
        pred_mag = finkus.func_hg1g2_with_spin(
            pha, H, G1, G2, R, np.radians(ra0), np.radians(dec0)
        ) + dist_reduction(eph["Dobs"], eph["Dhelio"])
        ax[fi].plot(eph["Date"], pred_mag, color=colors[fi],
                    label=r"$\texttt{sHG}_{\texttt{1}}\texttt{G}_{\texttt{2}}$")

    # Plot Residuals
    if res_sHG1G2:
        pha = np.transpose(
            [
                [i, j, k]
                for i, j, k in zip(
                    np.radians(ztf.loc[cond, "Phase"]),
                    np.radians(ztf.loc[cond, "RA"]),
                    np.radians(ztf.loc[cond, "Dec"]),
                )
            ]
        )
        pred_mag = finkus.func_hg1g2_with_spin(
            pha, H, G1, G2, R, np.radians(ra0), np.radians(dec0)
        )
        ax[2].scatter(
            ztf.loc[cond, "Date"],
            ztf.loc[cond, "i:magpsf"]
            - dist_reduction(ztf.loc[cond, "Dobs"], ztf.loc[cond, "Dhelio"])
            - pred_mag,
            color=colors[fi],
            s=marker_size,
            label=r"$\texttt{sHG}_{\texttt{1}}\texttt{G}_{\texttt{2}}$"+ f" {filt}",
            rasterized=True
        )

        # Show influence of s(ra,dec)
        # geo = finkus.spin_angle(np.radians(radec.ra.deg),
        #             np.radians(radec.dec.deg), np.radians(ra0), np.radians(dec0))
        # funcs = 1 - (1 - R) * np.abs(geo)
        # funcs = 2.5 * np.log10(funcs)
        # ax[2].plot(
        #     eph["Date"], funcs)


    # ax[fi].set_ylabel(f'Apparent magnitude in {filt}')
    ax[fi].legend(ncol=2)  # loc='upper right')


# --------------------------------------------------------------------------------
# Residuals
ax[2].axhline(0, color="lightgray", zorder=-100)
ax[2].set_ylim(-0.6, 0.6)
ax[2].legend(ncol=4, loc="upper center")
ax[2].set_ylabel("Residuals")


# --------------------------------------------------------------------------------
# Axes
fig.text(0.01, 0.63, "Apparent magnitude", va="center", rotation="vertical")
ax[2].set_xlabel("Time / days")

fig.savefig(
    os.path.join(data_fink, "gfx", "article", f"{target}-time.png"), facecolor="white"
)
# fig.savefig(f'{data_fink}/gfx_obs/{target}-time.pgf', facecolor='white')
# plt.show()

HG       : 10.02  0.00
HG1G2    : 10.18  0.82  0.00
HG21G1  g: 10.65  0.61  0.20   228.3  44.8   0.10    0.068
HG21G1  r: 10.07  0.37  0.31   228.3  44.8   0.10    0.068


In [73]:
sc = rocks.Rock( target )

print("   RA  DEC    sep s/u --   alt s/u")
ra0_unc = data.loc[data.ssnamenr == target, "SHG1G2_dalpha0"].values[0]
dec0_unc = data.loc[data.ssnamenr == target, "SHG1G2_ddelta0"].values[0]
unc_fink = np.sqrt( ra0_unc**2 + dec0_unc**2 )

for i,s in enumerate(sc.parameters.physical.spin):
    ang_sep = np.degrees(angular_separation( np.radians(s.RA0.value), np.radians(s.DEC0.value),
                                  np.radians(ra0), np.radians(dec0)
                                 ))
    ang_sep_alt = np.degrees(angular_separation( np.radians(s.RA0.value), np.radians(s.DEC0.value),
                                  np.radians(data.loc[data.ssnamenr == target, "SHG1G2_alpha0_alt"].values[0]), np.radians(data.loc[data.ssnamenr == target, "SHG1G2_delta0_alt"].values[0])
                                 ))
    unc_spin = np.sqrt( s.RA0.error_**2 + s.DEC0.error_**2 )

    unc = np.sqrt( unc_fink**2 + unc_spin**2 )
    print( f"{i:2d} {s.RA0.value:3.0f} {s.DEC0.value:3.0f}  {ang_sep:5.1f} {ang_sep/unc:3.1f} -- {ang_sep_alt:5.1f} {ang_sep_alt/unc:3.1f}" )

   RA  DEC    sep s/u --   alt s/u
 0  13  25  103.0 4.0 --  77.0 3.0
 1 211  15   33.0 1.4 -- 147.0 6.1


In [74]:
data.columns

Index(['ssnamenr', 'HG_chi2red', 'HG_status', 'HG_fit', 'HG_rms', 'HG_rms_g',
       'HG_rms_r', 'HG_median_error_phot', 'HG_median_error_phot_1',
       'HG_median_error_phot_2', 'HG_H_g', 'HG_dH_g', 'HG_G_g', 'HG_dG_g',
       'HG_H_r', 'HG_dH_r', 'HG_G_r', 'HG_dG_r', 'HG_flag', 'HG1G2_chi2red',
       'HG1G2_status', 'HG1G2_fit', 'HG1G2_rms', 'HG1G2_rms_g', 'HG1G2_rms_r',
       'HG1G2_median_error_phot', 'HG1G2_median_error_phot_1',
       'HG1G2_median_error_phot_2', 'HG1G2_H_g', 'HG1G2_dH_g', 'HG1G2_G1_g',
       'HG1G2_dG1_g', 'HG1G2_G2_g', 'HG1G2_dG2_g', 'HG1G2_H_r', 'HG1G2_dH_r',
       'HG1G2_G1_r', 'HG1G2_dG1_r', 'HG1G2_G2_r', 'HG1G2_dG2_r', 'HG1G2_flag',
       'SHG1G2_chi2red', 'SHG1G2_min_cos_lambda', 'SHG1G2_mean_cos_lambda',
       'SHG1G2_max_cos_lambda', 'SHG1G2_status', 'SHG1G2_fit', 'SHG1G2_rms',
       'SHG1G2_rms_g', 'SHG1G2_rms_r', 'SHG1G2_median_error_phot',
       'SHG1G2_median_error_phot_1', 'SHG1G2_median_error_phot_2', 'n_obs',
       'n_obs_g', 'n_obs_r', 